Data Processing for Tethered Balloon

In [83]:
# imports
import math
import numpy as np
import plotly as plt
import pandas as pd
from datetime import datetime
from scipy import signal
from scipy.fft import fft

In [84]:
# Constants + Strain Gauge Calibration
offset_0 = 406.62
offset_1 = -466.48
offset_2 = 246.45
offset_3 = 313.22

scale_0 = 0.001
scale_1 = 0.001
scale_2 = 0.001
scale_3 = 0.001

duty_cycle = 40
net_or_not = "no_net"

data_folder = f"../four_tether_{net_or_not}_7_1"
filename = f"{data_folder}/four_tether_{duty_cycle}%_{net_or_not}.csv"
print(f"Loading data from: {filename}")

Loading data from: ../four_tether_no_net_7_1/four_tether_40%_no_net.csv


In [85]:
def load_tether_data(filepath):
    """Load CSV data and apply calibration and alignment."""
    
    df = pd.read_csv(filepath, header=None, names=['time', 'raw0', 'raw1', 'raw2', 'raw3'])
    
    df['T0'] = ((df['raw0'] * scale_0) + offset_0) * math.sin(math.radians(55))
    df['T1'] = ((df['raw1'] * scale_1) + offset_1) * math.sin(math.radians(55))
    df['T2'] = ((df['raw2'] * scale_2) + offset_2) * math.sin(math.radians(55))
    df['T3'] = ((df['raw3'] * scale_3) + offset_3) * math.sin(math.radians(55))
    
    for col in ['T0', 'T1', 'T2', 'T3']:
        df[col] = df[col] - df[col].min()

    df['time'] = (df['time'] - df['time'][0])/1000

    df = df[['time', 'T0', 'T1', 'T2', 'T3']]
    
    return df

In [86]:
# Load and calibrate data
df = load_tether_data(filename)
print(f"Loaded data for {duty_cycle}% duty cycle, {net_or_not}")
print(df.head())

Loaded data for 40% duty cycle, no_net
    time         T0        T1         T2        T3
0  0.000  11.893269  0.403023  10.490880  0.497225
1  0.108  11.378022  1.046057   9.869144  1.451537
2  0.216  10.946329  1.084557   9.417791  1.395016
3  0.324  11.258426  0.824886   9.734803  1.158281
4  0.432  11.344437  0.922365   9.638962  1.288526


In [87]:
def plot_tether_tension(df, title=None):
    """Plot tether tension data over time."""
    
    import plotly.graph_objects as go
    
    fig = go.Figure()
    
    tension_columns = [col for col in df.columns if col != 'time']
    
    for col in tension_columns:
        fig.add_trace(go.Scatter(
            x=df['time'],
            y=df[col],
            mode='lines',
            name=col,
            line=dict(width=2)
        ))
    
    fig.update_layout(
        title=title or f"{duty_cycle}% Duty Cycle - {net_or_not.replace('_', ' ').title()}",
        xaxis_title='Time (s)',
        yaxis_title='Tension (g)',
        template='plotly_white',
        hovermode='x unified',
        height=500,
        width=1000
    )
    
    fig.show()

In [88]:
# Plot the tether tension data
plot_tether_tension(df)

Find Vortex Shedding Frequency and Analyze

In [89]:
# Fourier Transform Analysis
from scipy.signal import find_peaks

def analyze_frequency_spectrum(df, column, title="Frequency Analysis"):
    """Perform FFT analysis on a tether column to find dominant frequencies."""
    import plotly.graph_objects as go
    
    # Get data and time sampling
    data = df[column].values
    time = df['time'].values
    N = len(data)
    
    # Time sampling interval
    dt = np.mean(np.diff(time))
    fs = 1 / dt  # Sampling frequency
    
    # Perform FFT
    fft_vals = fft(data)
    mag = np.abs(fft_vals[:N//2]) * 2/N  # Magnitude spectrum
    freq = np.fft.fftfreq(N, dt)[:N//2]  # Frequency axis
    phase = np.angle(fft_vals[:N//2])  # Phase spectrum
    
    # Find peaks in magnitude spectrum (exclude DC component)
    peaks, properties = find_peaks(mag[1:], height=np.max(mag)*0.05)
    peaks = peaks + 1  # Adjust for DC component
    
    # Sort by magnitude and get top frequencies
    top_peaks = peaks[np.argsort(properties['peak_heights'])[-3:]]
    top_freqs = freq[top_peaks]
    top_mags = mag[top_peaks]
    
    # Create figure
    fig = go.Figure()
    
    # Add magnitude spectrum
    fig.add_trace(go.Scatter(
        x=freq,
        y=mag,
        mode='lines',
        name='Magnitude',
        line=dict(color='blue')
    ))
    
    # Mark peaks
    fig.add_trace(go.Scatter(
        x=freq[top_peaks],
        y=mag[top_peaks],
        mode='markers',
        name='Peaks',
        marker=dict(size=8, color='red')
    ))
    
    fig.update_layout(
        title=f'{title} - {column} ({duty_cycle}%, {net_or_not.replace("_", " ").title()})',
        xaxis_title='Frequency (Hz)',
        yaxis_title='Magnitude',
        template='plotly_white',
        hovermode='closest',
        height=500,
        width=1000
    )
    fig.show()
    
    return {
        'frequencies': freq,
        'magnitude': mag,
        'phase': phase,
        'fs': fs,
        'dt': dt,
        'top_frequencies': top_freqs,
        'top_magnitudes': top_mags
    }

# Run FFT analysis on all tethers
fft_results = {}
for col in ['T0', 'T1', 'T2', 'T3']:
    fft_results[col] = analyze_frequency_spectrum(df, col)
    print(f"\n{col} - Top Frequencies: {fft_results[col]['top_frequencies'][:3]} Hz")
    print(f"Magnitudes: {fft_results[col]['top_magnitudes'][:3]}")


T0 - Top Frequencies: [1.26028807 1.05452675 1.18312757] Hz
Magnitudes: [2.4072636  2.58809498 3.75323779]



T1 - Top Frequencies: [0.12860082 0.25720165 0.20576132] Hz
Magnitudes: [6.22356012 7.23428122 8.84888723]



T2 - Top Frequencies: [1.28600823 0.12860082 0.95164609] Hz
Magnitudes: [1.21527907 1.31574467 1.61716195]



T3 - Top Frequencies: [0.28292181 0.23148148 0.05144033] Hz
Magnitudes: [0.38059167 0.38858274 0.4148682 ]


In [90]:
# Compute Strouhal Number
def compute_strouhal(vortex_freq, characteristic_length, wind_velocity):
    """
    Calculate Strouhal number: St = f * D / V
    
    Parameters:
    -----------
    vortex_freq : float
        Vortex shedding frequency (Hz)
    characteristic_length : float
        Characteristic length (balloon diameter) in meters
    wind_velocity : float
        Wind velocity in m/s
        
    Returns:
    --------
    float : Strouhal number
    """
    return (vortex_freq * characteristic_length) / wind_velocity

# Parameters (adjust these based on your setup)
balloon_diameter = 0.447  # meters - adjust based on your balloon size
wind_velocity = 5.0  # m/s - adjust based on your wind speed

# Calculate Strouhal for dominant frequency of each tether
print("\n" + "="*60)
print(f"Strouhal Number Analysis - {duty_cycle}% {net_or_not.replace('_', ' ').title()}")
print("="*60)

for col in ['T0', 'T1', 'T2', 'T3']:
    dominant_freq = fft_results[col]['top_frequencies'][0]
    st = compute_strouhal(dominant_freq, balloon_diameter, wind_velocity)
    print(f"{col}: Frequency = {dominant_freq:.2f} Hz, Strouhal = {st:.4f}")
    
print("\nNote: Typical Strouhal number for circular cylinders ≈ 0.2")


Strouhal Number Analysis - 40% No Net
T0: Frequency = 1.26 Hz, Strouhal = 0.1127
T1: Frequency = 0.13 Hz, Strouhal = 0.0115
T2: Frequency = 1.29 Hz, Strouhal = 0.1150
T3: Frequency = 0.28 Hz, Strouhal = 0.0253

Note: Typical Strouhal number for circular cylinders ≈ 0.2
